<!--
Copyright (c) 2026 Milin Patel
Hochschule Kempten - University of Applied Sciences
Campustage 2026 - Automatisiertes Fahren! Sicher durch Zufall?
MIT License - see LICENSE.
-->

# Der KI-Blick
### Wie sieht ein selbstfahrendes Auto, und wann sieht es nichts?

**Campustage 2026 · Automatisiertes Fahren! Sicher durch Zufall?**<br>
Milin Patel · Hochschule Kempten, Institut für Fahrerassistenz und vernetzte Mobilität (IFM)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/01_Perception_Systems/notebooks/00_workshop_der_ki_blick.ipynb)

---

### Deine Rolle heute

Du arbeitest im Entwicklungsteam eines selbstfahrenden Autos. Das Auto hat eine Kamera, und eine KI soll darin Menschen erkennen.
Vor dir liegen sieben Missionen. Am Ende entscheidest du selbst: **Darf dein Auto auf die Straße?**

Du musst dafür nichts vorher wissen. Probier aus, rate, lass dich überraschen.

### So startest du

1. Oben im Menü: **Laufzeit → Alle ausführen** (englisches Menü: *Runtime → Run all*).
2. Falls Colab fragt, ob du dem Notebook vertraust: bestätigen und fortfahren.
3. Kurz warten, bis unter **Start** ein grünes **Bereit** erscheint (ca. 1 Minute).
4. Dann nach unten scrollen. Alles läuft über Knöpfe und Regler. Code musst du nicht lesen.

> **Ein Wort von mir vorab:** Ich forsche an der Hochschule Kempten daran, wie man KI im Auto sicher macht.
> Am meisten lerne ich in den Momenten, in denen die KI etwas tut, womit niemand gerechnet hat.
> Genau solche Momente kannst du heute selbst auslösen. *Milin*

In [ ]:
# @title Start: KI laden (einfach abwarten, bis „Bereit“ erscheint)
%pip install -q ultralytics==8.4.216
import io, time
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import ipywidgets as W
from IPython.display import display
from ultralytics import YOLO
from ultralytics.utils import ASSETS

try:
    from google.colab import output as _colab
    _colab.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ---------- Farben, Texte ----------
ORANGE, BLAU, GRAU, LILA = (225, 120, 0), (40, 110, 200), (105, 105, 115), (120, 60, 170)
DE = {"person": "Mensch", "car": "Auto", "bus": "Bus", "truck": "LKW", "bicycle": "Fahrrad",
      "motorcycle": "Motorrad", "stop sign": "Stoppschild", "traffic light": "Ampel",
      "train": "Zug", "handbag": "Handtasche", "backpack": "Rucksack", "umbrella": "Schirm",
      "skateboard": "Skateboard", "dog": "Hund", "cat": "Katze", "bird": "Vogel",
      "tie": "Krawatte", "cell phone": "Handy", "bench": "Bank", "chair": "Stuhl",
      "fire hydrant": "Hydrant", "parking meter": "Parkuhr", "sports ball": "Ball",
      "suitcase": "Koffer", "bottle": "Flasche", "cup": "Becher", "laptop": "Laptop",
      "potted plant": "Pflanze", "tennis racket": "Tennisschläger", "baseball bat": "Schläger",
      "horse": "Pferd", "kite": "Drachen", "clock": "Uhr", "book": "Buch", "tv": "Bildschirm"}
def de(name): return DE.get(name, name)
def prozent(x): return f"{int(100 * x + 1e-6)} %"   # abrunden: 24,97 % wird 24 %, nicht 25 %
def sek(x): return f"{x:.2f}".replace(".", ",") + " s"

def karte(text, art="info"):
    farben = {"info": ("#eef4fb", "#1d4e89"), "gut": ("#e9f6ee", "#1e6b3a"),
              "achtung": ("#fdf0e4", "#8a4b0f"), "frage": ("#f3eefa", "#5b2d8a")}
    bg, fg = farben[art]
    return W.HTML(f'<div style="background:{bg};color:{fg};border-left:6px solid {fg};'
                  f'padding:10px 14px;margin:6px 0;border-radius:6px;font-size:15px;'
                  f'line-height:1.5;font-family:sans-serif">{text}</div>')

def kacheln(werte):
    """werte: Liste von (Titel, Zahl, Farbe)."""
    html = "".join(
        f'<div style="flex:1;min-width:110px;background:{f};color:white;border-radius:8px;'
        f'padding:8px 10px;text-align:center;font-family:sans-serif">'
        f'<div style="font-size:13px">{t}</div><div style="font-size:30px;font-weight:bold">{z}</div></div>'
        for t, z, f in werte)
    return W.HTML(f'<div style="display:flex;gap:8px;flex-wrap:wrap;margin:6px 0">{html}</div>')

def knopf(text, stil="primary", breite="240px"):
    return W.Button(description=text, button_style=stil, layout=W.Layout(width=breite, height="38px"))

def sicher(ausgabe):
    """Fehler in Knopf-Funktionen freundlich anzeigen statt still zu verschlucken."""
    def deko(fn):
        def innen(*a, **k):
            try:
                return fn(*a, **k)
            except Exception as e:
                with ausgabe:
                    display(karte(f"Hoppla, hier ist etwas schiefgelaufen ({type(e).__name__}: {e}). "
                                  "Führe diese Zelle nochmal aus (▶ links) oder frag kurz nach.", "achtung"))
        return innen
    return deko

# ---------- Bilder ----------
def arbeitsbild(img):
    """Auf 640 Pixel (lange Seite) bringen, Seiten durch 32 teilbar."""
    img = img.convert("RGB")
    f = 640 / max(img.size)
    w = max(64, int(round(img.width * f / 32)) * 32)
    h = max(64, int(round(img.height * f / 32)) * 32)
    return img.resize((w, h), Image.LANCZOS)

def jpg(img):
    b = io.BytesIO(); img.save(b, "JPEG", quality=88); return b.getvalue()

def zeige(img, breite="520px"):
    return W.Image(value=jpg(img), format="jpeg", layout=W.Layout(width="100%", max_width=breite))

def nebeneinander(*teile):
    return W.HBox([W.VBox([W.HTML(f'<b style="font-family:sans-serif;font-size:15px">{t}</b>'),
                           zeige(b, "420px")], layout=W.Layout(width=f"{96 // len(teile)}%"))
                   for t, b in teile])

def _schrift(groesse):
    for n in ("DejaVuSans-Bold.ttf", "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
              "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
              "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf"):
        try: return ImageFont.truetype(n, groesse)
        except Exception: pass
    try: return ImageFont.load_default(size=groesse)
    except TypeError: return ImageFont.load_default()

def _gestrichelt(d, box, farbe, lw, l=9):
    x1, y1, x2, y2 = box
    for (ax, ay), (bx, by) in (((x1, y1), (x2, y1)), ((x2, y1), (x2, y2)), ((x2, y2), (x1, y2)), ((x1, y2), (x1, y1))):
        n = max(2, int(max(abs(bx - ax), abs(by - ay)) / l))
        for i in range(0, n, 2):
            a, b = i / n, min(1, (i + 1) / n)
            d.line([(ax + (bx - ax) * a, ay + (by - ay) * a), (ax + (bx - ax) * b, ay + (by - ay) * b)], fill=farbe, width=lw)

def _etikett(d, x, y, text, farbe, f):
    l, t, r, b = d.textbbox((0, 0), text, font=f)
    th = b - t
    x = max(0, min(x, d.im.size[0] - (r - l) - 8))
    y = y - th - 8 if y - th - 8 >= 0 else y + 2
    d.rectangle([x, y, x + (r - l) + 8, y + th + 6], fill=farbe)
    d.text((x + 4, y + 3 - t), text, fill=(255, 255, 255), font=f)

def zeichne(img, boxen, grenze, vermutungen=False, nummern=None, nur_menschen=False):
    """Boxen über der Grenze: durchgezogen. Darunter (wenn gewünscht): grau gestrichelt."""
    out = img.copy(); d = ImageDraw.Draw(out)
    f = _schrift(max(13, img.width // 34)); lw = max(2, img.width // 170)
    for b in sorted(boxen, key=lambda b: b["conf"]):
        if nur_menschen and b["name"] != "person":
            continue
        drin = b["conf"] >= grenze
        if not drin and not vermutungen:
            continue
        box = [int(v) for v in b["xyxy"]]
        if drin:
            farbe = ORANGE if b["name"] == "person" else BLAU
            d.rectangle(box, outline=farbe, width=lw)
        else:
            farbe = GRAU
            _gestrichelt(d, box, farbe, max(1, lw - 1))
        _etikett(d, box[0], box[1], f"{de(b['name'])} {prozent(b['conf'])}", farbe, f)
    if nummern:
        fg = _schrift(max(22, img.width // 14))
        for i, b in enumerate(nummern, 1):
            x1, y1, x2, y2 = b["xyxy"]
            cx, cy = (x1 + x2) / 2, y1 + (y2 - y1) * 0.15
            r = max(16, img.width // 26)
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=LILA, outline=(255, 255, 255), width=3)
            d.text((cx, cy), str(i), fill=(255, 255, 255), font=fg, anchor="mm")
    return out

# ---------- KI ----------
MODELL = YOLO("yolov8n.pt")
NAMEN = MODELL.names
STANDARD = 0.30
_cache = {}

def erkenne_alles(img):
    """Einmal mit sehr niedriger Grenze rechnen; danach wird nur noch gefiltert (darum ist der Regler sofort da)."""
    key = hash(img.tobytes())
    if key not in _cache:
        if len(_cache) > 60: _cache.clear()
        t = time.perf_counter()
        r = MODELL.predict(img, conf=0.05, imgsz=640, verbose=False)[0]
        dauer = time.perf_counter() - t
        boxen = [dict(name=NAMEN[int(b.cls[0])], conf=float(b.conf[0]), xyxy=[float(v) for v in b.xyxy[0]])
                 for b in r.boxes]
        _cache[key] = (boxen, dauer)
    return _cache[key]

def menschen(boxen, grenze):
    return [b for b in boxen if b["name"] == "person" and b["conf"] >= grenze]

def iou(a, b):
    ax1, ay1, ax2, ay2 = a; bx1, by1, bx2, by2 = b
    iw, ih = max(0, min(ax2, bx2) - max(ax1, bx1)), max(0, min(ay2, by2) - max(ay1, by1))
    i = iw * ih
    return i / ((ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - i + 1e-9)

def bester_treffer(boxen, ziel):
    """Höchste Konfidenz eines Menschen an der Stelle des Ziels (0, wenn dort keiner mehr ist)."""
    werte = [b["conf"] for b in boxen if b["name"] == "person" and iou(b["xyxy"], ziel["xyxy"]) > 0.3]
    return max(werte, default=0.0)

S = dict(bild=None, quelle=None, gezaehlt=None, zeit_mensch=None, zeit_ki=None, ki_menschen=None,
         grenze=None, schlimmer=None, wetter_tipp=None, wetter_ergebnis=None, aufkleber=None,
         quiz={}, antwort=None, freigabe=None, wunsch="")

def grenze(): return S["grenze"] if S["grenze"] is not None else STANDARD

def neues_bild(img, quelle):
    S.update(bild=arbeitsbild(img), quelle=quelle, gezaehlt=None, zeit_mensch=None, zeit_ki=None,
             ki_menschen=None, wetter_ergebnis=None, aufkleber=None)

# ---------- Wetter (simuliert) ----------
def wetter(img, art, staerke):
    """staerke von 0 bis 1."""
    if staerke <= 0 or art == "Klar":
        return img
    rng = np.random.default_rng(7)
    if art == "Nebel":
        grau = Image.new("RGB", img.size, (208, 208, 212))
        return Image.blend(img, grau, 0.88 * staerke).filter(ImageFilter.GaussianBlur(0.5 + 2.5 * staerke))
    if art == "Regen":
        o = img.copy(); d = ImageDraw.Draw(o); w, h = img.size
        lang = max(8, h // 30)
        for _ in range(int(w * h / 150 * staerke)):
            x, y = int(rng.integers(0, w)), int(rng.integers(0, h))
            d.line([(x, y), (x - lang // 5, y + lang)], fill=(195, 200, 215), width=2)
        o = o.filter(ImageFilter.GaussianBlur(0.6 + 1.2 * staerke))
        return Image.eval(o, lambda p: int(p * (1 - 0.35 * staerke)))
    if art == "Dunkelheit":
        a = np.asarray(img).astype(np.float32) * (1 - 0.93 * staerke)
        a += rng.normal(0, 18 * staerke, a.shape)          # Kamera-Rauschen wie bei echten Nachtbildern
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    return img

# ---------- Aufkleber ----------
def aufkleber_ort(img, ziel, anteil):
    x1, y1, x2, y2 = ziel["xyxy"]
    g = int(max(12, anteil * (y2 - y1)))
    g = min(g, img.width, img.height)
    cx = int((x1 + x2) / 2); cy = int(y1 + 0.38 * (y2 - y1))
    px = min(max(0, cx - g // 2), img.width - g); py = min(max(0, cy - g // 2), img.height - g)
    return px, py, g

def zufalls_aufkleber(img, ziel, anteil, rng):
    px, py, g = aufkleber_ort(img, ziel, anteil)
    muster = Image.fromarray(rng.integers(0, 256, (g, g, 3), dtype=np.uint8))
    o = img.copy(); o.paste(muster, (px, py)); return o, muster

def berechne_aufkleber(img, ziel, anteil, schritte=40, fortschritt=None):
    """Gezielter Angriff: Das Muster wird Schritt für Schritt so verändert,
    dass die KI an dieser Stelle immer weniger 'Mensch' sieht."""
    px, py, g = aufkleber_ort(img, ziel, anteil)
    net = MODELL.model.eval()
    dev = next(net.parameters()).device
    dtype = next(net.parameters()).dtype
    for p in net.parameters(): p.requires_grad_(False)
    x0 = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev)
    x1, y1, x2, y2 = ziel["xyxy"]
    patch = torch.rand(1, 3, g, g, device=dev, requires_grad=True)
    for i in range(schritte):
        x = x0.clone(); x[:, :, py:py + g, px:px + g] = patch
        out = net(x.to(dtype)); out = out[0] if isinstance(out, (list, tuple)) else out
        bx, sc = out[0, :4].float(), out[0, 4].float()          # Zeile 4 = Klasse 'person'
        drin = (bx[0] > x1) & (bx[0] < x2) & (bx[1] > y1) & (bx[1] < y2)
        wert = sc[drin].max() if bool(drin.any()) else sc.max()
        wert.backward()
        with torch.no_grad():
            patch -= 0.03 * patch.grad.sign(); patch.clamp_(0, 1); patch.grad.zero_()
        if fortschritt is not None: fortschritt.value = i + 1
    muster = Image.fromarray((patch.detach()[0].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8))
    o = img.copy(); o.paste(muster, (px, py)); return o, muster

# ---------- Start ----------
neues_bild(Image.open(ASSETS / "bus.jpg"), "Beispielbild (Straßenszene)")
erkenne_alles(S["bild"]); _cache.clear(); erkenne_alles(S["bild"])   # Aufwärmen, damit die Zeitmessung fair ist
display(karte("<b>Bereit.</b> Die KI ist geladen. Scroll jetzt nach unten zu <b>Mission 1</b>.", "gut"))

---
## Mission 1 · Wodurch schaut dein Auto?

Die KI braucht ein Bild. Du kannst das Beispielbild nehmen, eine Straßenszene in Madrid.

**Noch besser:** Habt ihr vorhin mit dem Roboter ein Foto gemacht? Lade es hier hoch.
Dann schaut die KI durch die Augen eures Roboters. Am spannendsten sind Fotos, auf denen Menschen zu sehen sind.

In [ ]:
# @title Mission 1 · Bild auswählen
m1_out = W.Output()
m1_upload = W.FileUpload(accept="image/*", multiple=False, description="Foto wählen")
m1_nimm = knopf("Dieses Foto verwenden", breite="220px")
m1_std = knopf("Zurück zum Beispielbild", stil="", breite="220px")

def m1_zeigen(text, art="info"):
    with m1_out:
        m1_out.clear_output(wait=True)
        display(karte(text, art)); display(zeige(S["bild"]))

def _hochgeladen(u):
    v = u.value
    if not v: return None
    if isinstance(v, dict): return list(v.values())[0]["content"]
    return bytes(v[0]["content"])

@sicher(m1_out)
def m1_nimm_klick(_):
    daten = _hochgeladen(m1_upload)
    if daten is None:
        m1_zeigen("Noch kein Foto gewählt. Klick zuerst auf <b>Foto wählen</b>.", "achtung"); return
    neues_bild(Image.open(io.BytesIO(daten)), "dein Foto")
    m1_zeigen("<b>Dein Foto ist geladen.</b> Ab jetzt arbeitet die KI in allen Missionen mit diesem Bild. "
              "Mach in Mission 2 einfach weiter.", "gut")

@sicher(m1_out)
def m1_std_klick(_):
    neues_bild(Image.open(ASSETS / "bus.jpg"), "Beispielbild (Straßenszene)")
    m1_zeigen("Das Beispielbild ist wieder aktiv.")

m1_nimm.on_click(m1_nimm_klick); m1_std.on_click(m1_std_klick)
display(W.HBox([m1_upload, m1_nimm, m1_std]), m1_out)
m1_zeigen(f"Aktuelles Bild: <b>{S['quelle']}</b>. Du kannst so weitermachen oder ein eigenes Foto hochladen.")

---
## Mission 2 · Zähl-Duell: Du gegen die KI

Bevor die KI drankommt, bist du dran.

1. Drück auf **Start**. Dann siehst du das Bild.
2. Zähl so schnell und so genau wie möglich: **Wie viele Menschen sind auf dem Bild?**
3. Zahl eintragen und auf **Fertig** drücken.

In [ ]:
# @title Mission 2 · Zähl-Duell
m2_out = W.Output(); m2_bild = W.Output()
m2_start = knopf("Start: Ich zähle jetzt", breite="220px")
m2_zahl = W.BoundedIntText(value=0, min=0, max=99, description="Menschen:", layout=W.Layout(width="170px"))
m2_fertig = knopf("Fertig", stil="success", breite="120px")
_m2 = {"t0": None}

@sicher(m2_out)
def m2_start_klick(_):
    _m2["t0"] = time.perf_counter()
    with m2_bild:
        m2_bild.clear_output(wait=True); display(zeige(S["bild"]))
    with m2_out:
        m2_out.clear_output(); display(karte("Die Zeit läuft. Zähl alle Menschen!", "frage"))

@sicher(m2_out)
def m2_fertig_klick(_):
    S["gezaehlt"] = int(m2_zahl.value)
    S["zeit_mensch"] = time.perf_counter() - _m2["t0"] if _m2["t0"] else None
    _m2["t0"] = None
    zeit = f" in <b>{sek(S['zeit_mensch'])}</b>" if S["zeit_mensch"] else ""
    with m2_out:
        m2_out.clear_output()
        display(karte(f"Du hast <b>{S['gezaehlt']} Menschen</b> gezählt{zeit}. Merk dir das. "
                      "Jetzt ist die KI dran: weiter zu Mission 3.", "gut"))

m2_start.on_click(m2_start_klick); m2_fertig.on_click(m2_fertig_klick)
display(m2_start, m2_bild, W.HBox([m2_zahl, m2_fertig]), m2_out)

---
## Mission 3 · Jetzt schaut die KI hin

Drück auf den Knopf. Die KI zeichnet um alles, was sie erkennt, einen Kasten.
Die Zahl am Kasten heißt **Konfidenz**: So sicher ist sich die KI.

Hat sie dasselbe gesehen wie du?

In [ ]:
# @title Mission 3 · Die KI schaut hin
m3_out = W.Output()
m3_los = knopf("KI hinschauen lassen", breite="220px")
m3_vermutung = W.ToggleButton(value=False, description="Auch alle Vermutungen zeigen", icon="eye",
                              layout=W.Layout(width="260px", height="38px"), disabled=True)

def m3_zeichnen():
    boxen, dauer = erkenne_alles(S["bild"])
    ki = len(menschen(boxen, STANDARD))
    S["zeit_ki"], S["ki_menschen"] = dauer, ki
    with m3_out:
        m3_out.clear_output(wait=True)
        if S["gezaehlt"] is None:
            vergleich = (f"Die KI sieht <b>{ki} Menschen</b>, in {sek(dauer)}. "
                         "(Du hast in Mission 2 noch nicht gezählt. Probier's ruhig noch!)")
        else:
            zeit = f" in {sek(S['zeit_mensch'])}" if S["zeit_mensch"] else ""
            vergleich = (f"<b>Du:</b> {S['gezaehlt']} Menschen{zeit} &nbsp;·&nbsp; "
                         f"<b>KI:</b> {ki} Menschen in {sek(dauer)}<br>")
            if ki == S["gezaehlt"]:
                vergleich += "Gleich viele. Aber hat die KI auch <i>dieselben</i> Menschen gefunden wie du? Schau genau hin."
            else:
                vergleich += "Ihr seid euch nicht einig. Wer hat recht? Such den Menschen, den einer von euch anders sieht."
        display(karte(vergleich, "frage"))
        display(zeige(zeichne(S["bild"], boxen, STANDARD, vermutungen=m3_vermutung.value)))
        if m3_vermutung.value:
            leise = sorted([b for b in boxen if b["conf"] < STANDARD], key=lambda b: -b["conf"])
            liste = ", ".join(f"{de(b['name'])} {prozent(b['conf'])}" for b in leise[:8]) or "keine"
            display(karte(f"<b>Grau gestrichelt</b> ist alles, was die KI nur vermutet: {liste}.<br>"
                          "Die KI „sieht“ viel mehr, als sie zugibt. Vieles davon ist Unsinn, manches aber nicht. "
                          "Irgendwer muss entscheiden, <b>ab wie viel Prozent</b> eine Vermutung zählt. "
                          "Das machst du in Mission 4.", "achtung"))
        else:
            display(karte(f"Die KI ist sich nie ganz sicher. Sie rechnet für jede Stelle im Bild aus, wie sehr sie nach "
                          "„Mensch“, „Bus“ oder „Fahrrad“ aussieht. Gezeigt wird nur, was über <b>{prozent(STANDARD)}</b> liegt.<br>"
                          "Neugierig, was darunter liegt? Drück auf <b>Auch alle Vermutungen zeigen</b>.", "info"))

@sicher(m3_out)
def m3_klick(_):
    m3_vermutung.disabled = False; m3_zeichnen()

m3_los.on_click(m3_klick)
m3_vermutung.observe(lambda c: sicher(m3_out)(lambda: m3_zeichnen())(), names="value")
display(W.HBox([m3_los, m3_vermutung]), m3_out)

---
## Mission 4 · Du stellst die Grenze ein

Jedes Auto, das Menschen erkennt, braucht eine **Grenze**: Alles darüber gilt als erkannt, alles darunter wird ignoriert.
Fachleute nennen das den *Schwellenwert*.

**Deine Aufgabe:** Stell die Grenze so ein, dass dein Auto **keinen Menschen übersieht**,
aber auch **nicht wegen jedem Schatten bremst**. Wenn du zufrieden bist: **Entscheidung festhalten**.

In [ ]:
# @title Mission 4 · Die Grenze
m4_out = W.Output(); m4_info = W.Output(); m4_dilemma_out = W.Output()
m4_regler = W.IntSlider(value=int(STANDARD * 100), min=5, max=95, step=5, description="Grenze (%)", continuous_update=False,
                        style={"description_width": "90px"}, layout=W.Layout(width="420px"))
m4_fest = knopf("Entscheidung festhalten", stil="success", breite="230px")

def m4_zeichnen(*_):
    g = m4_regler.value / 100
    boxen, _d = erkenne_alles(S["bild"])
    ki = len(menschen(boxen, g))
    andere = sorted({de(b["name"]) for b in boxen if b["name"] != "person" and b["conf"] >= g})
    with m4_out:
        m4_out.clear_output(wait=True)
        if S["gezaehlt"] is None:
            display(kacheln([("Erkannte Menschen", ki, "#555")]))
            display(karte("Zähl in Mission 2, dann zeigt dir das Auto hier auch, wen es übersieht.", "info"))
        else:
            uebersehen, zuviel = max(0, S["gezaehlt"] - ki), max(0, ki - S["gezaehlt"])
            display(kacheln([("Erkannte Menschen", ki, "#555"),
                             ("Übersehen", uebersehen, "#c0392b" if uebersehen else "#27ae60"),
                             ("Zu viel erkannt", zuviel, "#d35400" if zuviel else "#27ae60")]))
            if uebersehen:
                display(karte(f"Achtung: Dein Auto würde gerade <b>{uebersehen} Mensch{'en' if uebersehen > 1 else ''}</b> nicht sehen.", "achtung"))
            elif zuviel:
                display(karte("Dein Auto sieht Menschen, wo keine sind. Es würde ohne Grund bremsen.", "achtung"))
            else:
                display(karte("Passt, jedenfalls bei diesem Bild und bei Sonnenschein. Ob das bei Nebel auch klappt? Das testest du in Mission 5.", "gut"))
        if andere:
            display(W.HTML(f'<span style="font-family:sans-serif">Außerdem über der Grenze: {", ".join(andere)}</span>'))
        display(zeige(zeichne(S["bild"], boxen, g)))

@sicher(m4_info)
def m4_fest_klick(_):
    g = m4_regler.value / 100; S["grenze"] = g
    boxen, _d = erkenne_alles(S["bild"]); ki = len(menschen(boxen, g))
    von = f" von {S['gezaehlt']}" if S["gezaehlt"] is not None else ""
    with m4_info:
        m4_info.clear_output()
        display(karte(f"<b>Festgehalten: Grenze {prozent(g)}.</b> Damit erkennt dein Auto hier {ki}{von} Menschen.<br><br>"
                      "Diese Entscheidung hat <b>nicht die KI</b> getroffen und <b>nicht der Zufall</b>. "
                      "<b>Du</b> hast sie getroffen. Im echten Auto ist genau das die Arbeit von Menschen in Entwicklungsteams, "
                      "und sie müssen gut begründen können, warum die Grenze genau dort liegt.", "gut"))
    with m4_dilemma_out:
        m4_dilemma_out.clear_output()
        a = knopf("Einen Menschen übersehen", stil="danger", breite="250px")
        b = knopf("Ohne Grund bremsen", stil="warning", breite="250px")
        antwort = W.Output()
        ende = ("<br><br>Es gibt keine Grenze, die beides perfekt verhindert. Deshalb verlassen sich echte Autos "
                "nicht nur auf eine Kamera: Sie haben zusätzlich Radar und LiDAR, die anders „sehen“.")
        def wahl(text, schluessel):
            S["schlimmer"] = schluessel
            with antwort:
                antwort.clear_output(); display(karte(text + ende, "info"))
        a.on_click(lambda _: wahl("Verständlich: Ein übersehener Mensch kann sterben. Aber Vorsicht: Ein Auto, das immer wieder grundlos "
                                  "voll bremst, kann Auffahrunfälle auslösen. Und niemand will damit fahren.", "übersehen"))
        b.on_click(lambda _: wahl("Ein Punkt, den viele vergessen: Grundloses Vollbremsen ist auch gefährlich, zum Beispiel für das Auto dahinter. "
                                  "Trotzdem lässt sich ein übersehener Mensch nie wiedergutmachen.", "bremsen"))
        display(karte("<b>Kniffliger Fall:</b> Senkst du die Grenze, übersieht dein Auto weniger Menschen, bremst aber öfter grundlos. "
                      "Hebst du sie, ist es umgekehrt. <b>Was ist schlimmer?</b>", "frage"),
                W.HBox([a, b]), antwort)

m4_regler.observe(lambda c: sicher(m4_out)(m4_zeichnen)(), names="value")
m4_fest.on_click(m4_fest_klick)
display(karte("<b>Wer stellt diese Grenze im echten Auto ein?</b> Nicht die KI. Ein Mensch. Gerade eben: du.", "frage"),
        W.HBox([m4_regler, m4_fest]), m4_info, m4_out, m4_dilemma_out)
m4_zeichnen()

---
## Mission 5 · Die Welt ist nicht sauber

Bis jetzt hatte dein Auto perfektes Wetter. Die Straße hat das nicht.
Dein Auto fährt ab jetzt mit **deiner Grenze** aus Mission 4.

Erst ein Tipp: **Was macht der KI am meisten zu schaffen?**

In [ ]:
# @title Mission 5 · Wetter
m5_out = W.Output(); m5_ergebnis = W.Output()
m5_tipp = W.ToggleButtons(options=["Nebel", "Regen", "Dunkelheit"], value="Nebel")
m5_tipp_ok = knopf("Mein Tipp steht", breite="160px")
m5_art = W.ToggleButtons(options=["Nebel", "Regen", "Dunkelheit"], value="Nebel")
m5_staerke = W.IntSlider(value=50, min=0, max=100, step=10, description="Stärke (%)", continuous_update=False,
                         style={"description_width": "90px"}, layout=W.Layout(width="380px"))
m5_auswerten = knopf("Auswerten: Was schadet am meisten?", stil="success", breite="300px")
m5_test = W.VBox([karte("Jetzt probier aus. Wähle ein Wetter und dreh an der Stärke. "
                        "Wenn du alles ausprobiert hast: <b>Auswerten</b>.", "info"),
                  W.HBox([m5_art, m5_staerke]), m5_out, m5_auswerten, m5_ergebnis],
                 layout=W.Layout(display="none"))

def m5_zeichnen(*_):
    g = grenze(); klar = S["bild"]; schlecht = wetter(klar, m5_art.value, m5_staerke.value / 100)
    bk, _a = erkenne_alles(klar); bs, _b = erkenne_alles(schlecht)
    with m5_out:
        m5_out.clear_output(wait=True)
        display(kacheln([("Klar: Menschen", len(menschen(bk, g)), "#2c7a4b"),
                         (f"{m5_art.value}: Menschen", len(menschen(bs, g)), "#555")]))
        display(nebeneinander(("Klar", zeichne(klar, bk, g)),
                              (f"{m5_art.value} {m5_staerke.value} %", zeichne(schlecht, bs, g))))

@sicher(m5_out)
def m5_tipp_klick(_):
    S["wetter_tipp"] = m5_tipp.value; m5_tipp.disabled = True; m5_tipp_ok.disabled = True
    m5_test.layout.display = "flex"; m5_zeichnen()

@sicher(m5_ergebnis)
def m5_auswerten_klick(_):
    g = grenze(); klar = S["bild"]; bk, _a = erkenne_alles(klar)
    ziele = menschen(bk, g)
    res = {}
    for art in ["Nebel", "Regen", "Dunkelheit"]:
        bs, _b = erkenne_alles(wetter(klar, art, 0.7))
        verloren = sum(1 for z in ziele if bester_treffer(bs, z) < g)
        verlust = sum(z["conf"] - bester_treffer(bs, z) for z in ziele)
        res[art] = (verloren, verlust)
    schlimmst = max(res, key=lambda a: res[a])
    S["wetter_ergebnis"] = {a: v[0] for a, v in res.items()} | {"schlimmst": schlimmst}
    with m5_ergebnis:
        m5_ergebnis.clear_output(wait=True)
        display(kacheln([(f"{a}: verloren", v[0], "#c0392b" if a == schlimmst else "#777") for a, v in res.items()]))
        if res[schlimmst][0] > 0:
            text = (f"Bei 70 % Stärke hat <b>{schlimmst}</b> deinem Auto am meisten geschadet"
                    f" (von {len(ziele)} Menschen gingen {res[schlimmst][0]} verloren). ")
        else:
            text = (f"Bei 70 % Stärke ist bei diesem Bild kein Mensch ganz verschwunden. "
                    f"Am stärksten gesunken ist die Konfidenz aber bei <b>{schlimmst}</b>. Probier mal 100 %! ")
        text += ("<b>Dein Tipp war richtig.</b> " if S["wetter_tipp"] == schlimmst
                 else f"Dein Tipp war <b>{S['wetter_tipp']}</b>. Überrascht? ")
        if res["Dunkelheit"][0] == 0 and schlimmst != "Dunkelheit" and max(v[0] for v in res.values()) > 0:
            text += ("<br><br>Spannend: Dunkelheit macht dieser KI hier wenig aus. Dir als Mensch schon! "
                     "Die KI sieht nicht schlechter als du. Sie sieht <b>anders</b>.")
        text += ("<br><br><i>Ehrlich gesagt:</i> Das Wetter hier ist am Computer nachgemacht. Echter Nebel, echte Nacht und "
                 "Scheinwerfer-Blendung sind noch tückischer. Darum testen Teams ihre Autos in tausenden Situationen, "
                 "auf Teststrecken und in Simulationen, bevor sie auf die Straße dürfen.")
        display(karte(text, "gut"))

m5_tipp_ok.on_click(m5_tipp_klick); m5_auswerten.on_click(m5_auswerten_klick)
for w in (m5_art, m5_staerke): w.observe(lambda c: sicher(m5_out)(m5_zeichnen)(), names="value")
display(W.HBox([m5_tipp, m5_tipp_ok]), m5_test)

---
## Mission 6 · Kann man die KI austricksen?

Stell dir vor, jemand will dein Auto **absichtlich** täuschen. Kein Hacker im Computer, nur ein bunter Aufkleber auf der Jacke.
Kann ein Mensch damit für die KI unsichtbar werden?

**Runde 1:** Ein Aufkleber mit **zufälligem** Muster. Drück so oft du willst. Vielleicht hast du Glück?<br>
**Runde 2:** Ein **berechneter** Aufkleber. Der Computer sucht das Muster, das die KI am meisten verwirrt.

In [ ]:
# @title Mission 6 · Aufkleber
m6_out = W.Output(); m6_ergebnis = W.Output()
m6_ziel = W.Dropdown(description="Ziel:", layout=W.Layout(width="200px"))
m6_groesse = W.IntSlider(value=30, min=10, max=45, step=5, description="Aufkleber (%)", continuous_update=False,
                         style={"description_width": "100px"}, layout=W.Layout(width="380px"))
m6_zufall = knopf("Runde 1: Zufallsaufkleber", stil="warning", breite="230px")
m6_rechnen = knopf("Runde 2: Aufkleber berechnen", stil="danger", breite="250px")
m6_speichern = knopf("Aufkleber speichern", stil="", breite="190px")
m6_balken = W.IntProgress(value=0, min=0, max=50, description="Rechne:", layout=W.Layout(width="380px", visibility="hidden"))
_m6 = {"ziele": [], "versuche": 0, "erfolge": 0, "rng": np.random.default_rng(), "bester": None, "muster": None}

def m6_vorbereiten():
    boxen, _d = erkenne_alles(S["bild"])
    ziele = sorted(menschen(boxen, grenze()), key=lambda b: b["xyxy"][0])
    _m6["ziele"] = ziele
    m6_ziel.options = [(f"Mensch {i}", i - 1) for i in range(1, len(ziele) + 1)] or [("kein Mensch erkannt", -1)]
    m6_ziel.value = 0 if ziele else -1
    with m6_out:
        m6_out.clear_output(wait=True)
        if ziele:
            display(karte(f"Wähle, welcher Mensch verschwinden soll. Die Größe des Aufklebers ist in Prozent der Körpergröße. "
                          f"Gilt deine Grenze aus Mission 4: <b>{prozent(grenze())}</b>.", "info"))
            display(zeige(zeichne(S["bild"], boxen, grenze(), nummern=ziele, nur_menschen=True)))
        else:
            display(karte("Auf diesem Bild erkennt die KI keinen Menschen. Nimm in Mission 1 ein anderes Foto.", "achtung"))

def _ergebnis(img_neu):
    z = _m6["ziele"][m6_ziel.value]
    bn, _d = erkenne_alles(img_neu); vorher, nachher = z["conf"], bester_treffer(bn, z)
    return bn, vorher, nachher, nachher < grenze()

@sicher(m6_ergebnis)
def m6_zufall_klick(_):
    if m6_ziel.value is None or m6_ziel.value < 0: return
    z = _m6["ziele"][m6_ziel.value]
    neu, _m = zufalls_aufkleber(S["bild"], z, m6_groesse.value / 100, _m6["rng"])
    bn, vorher, nachher, weg = _ergebnis(neu)
    _m6["versuche"] += 1; _m6["erfolge"] += int(weg)
    with m6_ergebnis:
        m6_ergebnis.clear_output(wait=True)
        display(kacheln([("Versuche", _m6["versuche"], "#555"), ("Unsichtbar gemacht", _m6["erfolge"], "#c0392b"),
                         ("Konfidenz vorher → jetzt", f"{int(100*vorher)} → {int(100*nachher)}", "#d35400")]))
        display(zeige(zeichne(neu, bn, grenze(), nur_menschen=True)))
        if weg:
            display(karte("Wow, Glückstreffer! Das passiert selten. Drück nochmal: Klappt es wieder?", "achtung"))
        else:
            display(karte("Die KI erkennt den Menschen trotzdem. Zufall reicht meistens nicht. Noch ein paar Versuche, dann Runde 2.", "info"))

@sicher(m6_ergebnis)
def m6_rechnen_klick(_):
    if m6_ziel.value is None or m6_ziel.value < 0: return
    z = _m6["ziele"][m6_ziel.value]; a = m6_groesse.value / 100
    m6_balken.layout.visibility = "visible"; m6_balken.value = 0; m6_rechnen.disabled = True
    try:
        neu, muster = berechne_aufkleber(S["bild"], z, a, schritte=m6_balken.max, fortschritt=m6_balken)
    finally:
        m6_rechnen.disabled = False; m6_balken.layout.visibility = "hidden"
    zuf, _m = zufalls_aufkleber(S["bild"], z, a, np.random.default_rng(1))
    bn, vorher, nachher, weg = _ergebnis(neu)
    bz, _v, nach_z, _w = _ergebnis(zuf)
    _m6["muster"] = muster
    if weg and (_m6["bester"] is None or m6_groesse.value < _m6["bester"]): _m6["bester"] = m6_groesse.value
    S["aufkleber"] = dict(groesse=m6_groesse.value, geschafft=weg, bester=_m6["bester"])
    with m6_ergebnis:
        m6_ergebnis.clear_output(wait=True)
        display(nebeneinander((f"Zufall: Mensch {prozent(nach_z)}", zeichne(zuf, bz, grenze(), nur_menschen=True)),
                              (f"Berechnet: Mensch {prozent(nachher)}", zeichne(neu, bn, grenze(), nur_menschen=True))))
        if weg:
            display(karte(f"<b>Der Mensch ist für die KI verschwunden</b> (von {prozent(vorher)} auf {prozent(nachher)}, deine Grenze liegt bei {prozent(grenze())}). "
                          "Für dich ist er natürlich noch da.<br><br>"
                          f"<b>Challenge:</b> Wie klein geht's? Dein kleinster wirksamer Aufkleber: <b>{_m6['bester']} %</b> der Körpergröße. "
                          "Wer im Raum schafft weniger?", "achtung"))
        else:
            display(karte(f"Knapp: Die Konfidenz ist von {prozent(vorher)} auf {prozent(nachher)} gefallen, liegt aber noch über deiner Grenze. "
                          "Mach den Aufkleber etwas größer und rechne nochmal.", "info"))
        display(karte("<b>Was ist da passiert?</b> Der Computer hat 50-mal nachgeschaut, wie die KI auf das Muster reagiert, "
                      "und es jedes Mal ein kleines Stück verbessert. Dafür musste er die KI genau kennen. "
                      "<b>Zufall trifft fast nie. Absicht schon.</b><br><br>"
                      "<i>Ehrlich gesagt:</i> Auf dem Bildschirm ist das leichter als auf der Straße. Ein gedrucktes Muster muss bei jedem Licht "
                      "und aus jedem Winkel wirken. Forschende haben aber gezeigt, dass gedruckte Aufkleber auf Verkehrsschildern "
                      "Erkennungssysteme täuschen können. Deshalb gibt es Teams für IT-Sicherheit, die solche Angriffe suchen, "
                      "bevor es andere tun.", "info"))

@sicher(m6_ergebnis)
def m6_speichern_klick(_):
    if _m6["muster"] is None:
        with m6_ergebnis: display(karte("Berechne erst einen Aufkleber (Runde 2).", "achtung")); return
    gross = _m6["muster"].resize((600, 600), Image.NEAREST); gross.save("mein_aufkleber.png")
    if IN_COLAB:
        from google.colab import files; files.download("mein_aufkleber.png")
    else:
        with m6_ergebnis: display(karte("Gespeichert als <code>mein_aufkleber.png</code>.", "gut"))

m6_zufall.on_click(m6_zufall_klick); m6_rechnen.on_click(m6_rechnen_klick); m6_speichern.on_click(m6_speichern_klick)
m6_neu = knopf("Bild neu laden", stil="", breite="150px"); m6_neu.on_click(lambda _: sicher(m6_out)(m6_vorbereiten)())
display(W.HBox([m6_ziel, m6_groesse, m6_neu]), m6_out, W.HBox([m6_zufall, m6_rechnen, m6_speichern]), m6_balken, m6_ergebnis)
m6_vorbereiten()

---
## Mission 7 · Mensch oder KI: Wer sieht was?

Du hast gesehen, wo die KI scheitert. Aber ist sie deshalb einfach schlechter als du?
Tipp bei jeder Situation: **Wer ist hier im Vorteil?**

In [ ]:
# @title Mission 7 · Mensch oder KI
FRAGEN = [
    ("Ein Kind am Straßenrand schaut einem Ball hinterher, der auf die Straße rollt. Wer ahnt zuerst, dass es gleich losrennt?",
     "Mensch", "Menschen lesen Absichten: Blicke, Körperhaltung, die ganze Situation. Für eine KI ist das bis heute schwer. Sie sieht vor allem, wo das Kind gerade steht."),
    ("Wer kann gleichzeitig nach vorne, nach hinten und zu beiden Seiten schauen?",
     "KI-Auto", "Ein automatisiertes Auto hat Kameras und Sensoren rundherum und wertet alle gleichzeitig aus. Du kannst immer nur in eine Richtung schauen."),
    ("Eine Polizistin winkt dich an einer roten Ampel durch. Wer versteht sicher, was gemeint ist?",
     "Mensch", "Handzeichen, Nicken, Blickkontakt: Für dich selbstverständlich, für automatisierte Autos eine echte Herausforderung."),
    ("Nach vier Stunden Autobahn: Wer ist noch genauso aufmerksam wie am Anfang?",
     "KI-Auto", "Eine KI wird nicht müde, schaut nicht aufs Handy und lässt sich nicht ablenken."),
    ("Ein Aufkleber, der dir kaum auffällt, lässt einen Menschen verschwinden. Wen kann man damit täuschen?",
     "KI-Auto", "Das hast du in Mission 6 selbst ausprobiert. Dich täuscht so ein Muster nicht."),
    ("Nachts, keine Straßenlaterne, ein Fußgänger in dunkler Kleidung. Wer bemerkt ihn eher?",
     "KI-Auto", "Ein LiDAR schickt eigenes Laserlicht aus und braucht deshalb keine Beleuchtung. Eine normale Kamera allein hätte es dagegen schwer."),
    ("Auf der Autobahn liegt plötzlich ein Sofa. So etwas hat noch niemand vorher gesehen. Wer kommt besser klar?",
     "Mensch", "Menschen können oft auch mit völlig Neuem umgehen, weil sie die Welt verstehen. Eine KI kennt vor allem, was in ihren Trainingsdaten vorkam. Seltene, unbekannte Situationen gehören zu den größten offenen Problemen."),
]
m7_stand = W.Output(); _m7 = {"richtig": 0, "beantwortet": 0, "Mensch": 0, "KI-Auto": 0}
zeilen = []
for nr, (frage, loesung, erklaerung) in enumerate(FRAGEN, 1):
    k1, k2 = knopf("Mensch", stil="", breite="120px"), knopf("KI-Auto", stil="", breite="120px")
    aus = W.Output()
    def antworten(wahl, k1=k1, k2=k2, aus=aus, loesung=loesung, erklaerung=erklaerung, nr=nr):
        k1.disabled = k2.disabled = True
        ok = wahl == loesung
        _m7["beantwortet"] += 1; _m7["richtig"] += int(ok); _m7[loesung] += 1; S["quiz"][nr] = ok
        with aus:
            display(karte(("<b>Richtig!</b> " if ok else f"<b>Hier hat eher: {loesung}.</b> ") + erklaerung, "gut" if ok else "info"))
        with m7_stand:
            m7_stand.clear_output()
            if _m7["beantwortet"] == len(FRAGEN):
                display(kacheln([("Punkte Mensch", _m7["Mensch"], "#2c7a4b"), ("Punkte KI-Auto", _m7["KI-Auto"], "#2a5d9f"),
                                 ("Du lagst richtig", f"{_m7['richtig']}/{len(FRAGEN)}", "#555")]))
                display(karte("<b>Kein klarer Sieger.</b> Das Auto sieht nicht besser oder schlechter als du. Es sieht <b>anders</b>. "
                              "Darum ist die spannende Frage nicht „Mensch oder KI?“, sondern: "
                              "Wie bringen wir die Stärken von beiden so zusammen, dass es sicher ist?", "frage"))
    k1.on_click(lambda _, f=antworten: f("Mensch")); k2.on_click(lambda _, f=antworten: f("KI-Auto"))
    zeilen.append(W.VBox([W.HTML(f'<div style="font-family:sans-serif;font-size:15px;margin-top:10px"><b>{nr}.</b> {frage}</div>'),
                          W.HBox([k1, k2]), aus]))
display(*zeilen, m7_stand)

---
## Finale · Sicher durch Zufall?

Eine letzte Frage. Denk an alles, was du heute gemacht hast.

In [ ]:
# @title Finale
f_out = W.Output(); f_protokoll = W.Output(); f_freigabe = W.Output()
wahl = [knopf(t, stil="", breite="160px") for t in ("Die KI selbst", "Der Zufall", "Ein Mensch")]
ANTWORT = {
    "Die KI selbst": "Verständlich, die KI rechnet ja. Aber die Grenze, ab der ihre Rechnung zählt, hast <b>du</b> gesetzt: in Mission 4, mit dem Regler.",
    "Der Zufall": "Der Zufall spielt mit: Nebel, Regen, ein Aufkleber. Aber wo die Grenze liegt und wie das Auto darauf reagiert, entscheiden <b>Menschen</b>. In Mission 4 warst das du.",
    "Ein Mensch": "<b>Genau.</b> Und zwar du, in Mission 4. Im echten Auto sind es Entwicklungsteams.",
}
def f_klick(b):
    S["antwort"] = b.description
    for k in wahl: k.disabled = True
    with f_out:
        f_out.clear_output(); display(karte(ANTWORT[b.description], "gut" if b.description == "Ein Mensch" else "info"))
        display(karte("<b>Sicher durch Zufall? Nein.</b> Der Zufall ist die Welt: Wetter, Licht, Dinge, mit denen keiner gerechnet hat. "
                      "Sicherheit entsteht durch die Entscheidungen von Menschen, die genau damit rechnen.", "frage"))
for k in wahl: k.on_click(f_klick)

def f_protokoll_klick(_):
    p = []
    if S["gezaehlt"] is not None: p.append(f"Du hast {S['gezaehlt']} Menschen gezählt" + (f", in {sek(S['zeit_mensch'])}." if S['zeit_mensch'] else "."))
    if S["ki_menschen"] is not None: p.append(f"Die KI hat {S['ki_menschen']} Menschen gefunden, in {sek(S['zeit_ki'])}.")
    if S["grenze"] is not None: p.append(f"Du hast die Grenze auf <b>{prozent(S['grenze'])}</b> gesetzt.")
    if S["schlimmer"]: p.append("Für dich ist schlimmer: " + ("einen Menschen zu übersehen." if S["schlimmer"] == "übersehen" else "ohne Grund zu bremsen."))
    if S["wetter_ergebnis"]: p.append(f"Am meisten geschadet hat: {S['wetter_ergebnis']['schlimmst']} (dein Tipp: {S['wetter_tipp']}).")
    if S["aufkleber"]: p.append("Du hast einen Menschen mit einem Aufkleber unsichtbar gemacht." if S["aufkleber"]["geschafft"]
                                 else "Dein berechneter Aufkleber hat die KI geschwächt, aber nicht ganz getäuscht.")
    if S["quiz"]: p.append(f"Mensch oder KI: {sum(S['quiz'].values())} von {len(S['quiz'])} richtig.")
    with f_protokoll:
        f_protokoll.clear_output()
        display(karte("<b>Dein Missions-Protokoll</b><ul>" + "".join(f"<li>{x}</li>" for x in p) + "</ul>" if p
                      else "Noch leer. Spiel erst ein paar Missionen.", "info"))
        ja, nein = knopf("Ja, es darf fahren", stil="success", breite="200px"), knopf("Noch nicht", stil="warning", breite="200px")
        wunsch = W.Textarea(placeholder="Was müsste dein Auto noch lernen? (freiwillig)", layout=W.Layout(width="520px", height="70px"))
        def frei(entscheidung):
            S["freigabe"] = entscheidung; S["wunsch"] = wunsch.value
            with f_freigabe:
                f_freigabe.clear_output()
                display(karte("Mutig. Dann musst du begründen können, warum es auch bei Nebel, in der Nacht und bei Angriffen sicher genug ist. "
                              "Genau so eine Begründung muss es geben, bevor ein Auto zugelassen wird."
                              if entscheidung == "ja" else
                              "Gute Entscheidung. Und das ist kein Grund aufzugeben: Genau an diesen Lücken arbeiten gerade sehr viele Menschen. "
                              "Das Problem ist nicht gelöst, aber es wird jeden Tag besser.", "gut"))
        ja.on_click(lambda _: frei("ja")); nein.on_click(lambda _: frei("nein"))
        display(karte("<b>Und jetzt du als Entwicklungsteam: Darf dein Auto so auf die Straße?</b>", "frage"), wunsch, W.HBox([ja, nein]))

f_pro = knopf("Mein Missions-Protokoll zeigen", breite="280px"); f_pro.on_click(f_protokoll_klick)
display(karte("<b>Wer hat heute entschieden, ab wann dein Auto etwas als Menschen erkennt?</b>", "frage"),
        W.HBox(wahl), f_out, f_pro, f_protokoll, f_freigabe)

---
## Und wer macht das alles im echten Leben?

Vielleicht hast du schon gehört, dass KI Jobs wegnimmt. Schau dir an, was **du** heute gemacht hast:

| Das hast du heute gemacht | Das machen im echten Leben |
|---|---|
| Die Grenze eingestellt und abgewogen | Entwicklerinnen und Entwickler für funktionale Sicherheit |
| Dein Auto bei Nebel, Regen und Nacht getestet | Test- und Simulationsteams |
| Die KI mit einem Aufkleber ausgetrickst | Fachleute für IT-Sicherheit |
| Mensch und KI verglichen | Forschung zu Sensoren und KI |
| Über die Freigabe entschieden | Prüforganisationen und Behörden |

Keine dieser Aufgaben erledigt die KI für sich selbst. Je mehr KI im Auto steckt,
desto wichtiger werden Menschen, die sie verstehen, prüfen und besser machen.

---

## Bonus-Level: Für alle, die nicht aufhören wollen

Das hier war nur ein kleiner Ausschnitt. Zwei Notebooks zum Weitermachen, wenn du Lust hast (auf Englisch und mit mehr Code):

- **Fußgänger erkennen, aber richtig:** [Pedestrian Detection](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/01_Perception_Systems/notebooks/05_pedestrian_detection.ipynb)
- **Mehr Tricks gegen KI:** [Adversarial Attacks](https://colab.research.google.com/github/milinpatel07/campustage-2026-sicher-durch-zufall/blob/main/02_Failure_Analysis/notebooks/04_adversarial_attacks.ipynb), die Fortsetzung von Mission 6

In [diesem Repository](https://github.com/milinpatel07/campustage-2026-sicher-durch-zufall) warten über 30 weitere Notebooks:
Sensoren, Sicherheitsnormen, Schutz vor Hackern, Zulassung. Vieles davon begegnet dir wieder, wenn du Informatik studierst,
im Bachelor und noch tiefer im Master. Du musst heute nichts davon können. Gut zu wissen, dass es da ist.

---

*Danke fürs Mitmachen.* Milin Patel · Prof. Rolf Jung · Hochschule Kempten